In [ ]:
import os
import hashlib
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

train_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "train.csv"), usecols=["full_text", "label"])
val_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "val.csv"), usecols=["full_text", "label"])
test_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "test.csv"), usecols=["full_text", "label"])

def md5_hash(s):
    return hashlib.md5(s.encode("utf-8", errors="ignore")).hexdigest()

train_hash = set(train_df["full_text"].astype(str).map(md5_hash))
val_hash = set(val_df["full_text"].astype(str).map(md5_hash))
test_hash = set(test_df["full_text"].astype(str).map(md5_hash))

overlap_train_val = len(train_hash.intersection(val_hash))
overlap_train_test = len(train_hash.intersection(test_hash))
overlap_val_test = len(val_hash.intersection(test_hash))

print("Overlap train vs val:", overlap_train_val)
print("Overlap train vs test:", overlap_train_test)
print("Overlap val vs test:", overlap_val_test)


In [ ]:
import os
import hashlib
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

RAW_CSV_PATH = os.path.join(DATA_RAW_DIR, "WELFake_Dataset.csv")

df = pd.read_csv(RAW_CSV_PATH)

df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df["title"] = df["title"].fillna("").astype(str)
df["text"] = df["text"].fillna("").astype(str)

df["full_text"] = (df["title"].str.strip() + ". " + df["text"].str.strip()).str.strip()
df = df[df["full_text"].str.len() > 20].copy()
df["label"] = df["label"].astype(int)

def md5_hash(s):
    return hashlib.md5(s.encode("utf-8", errors="ignore")).hexdigest()

df["hash"] = df["full_text"].map(md5_hash)
df = df.drop_duplicates(subset=["hash"]).reset_index(drop=True)

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=42,
    stratify=df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

train_out = train_df[["full_text", "label"]].copy()
val_out = val_df[["full_text", "label"]].copy()
test_out = test_df[["full_text", "label"]].copy()

train_path = os.path.join(DATA_PROCESSED_DIR, "train.csv")
val_path = os.path.join(DATA_PROCESSED_DIR, "val.csv")
test_path = os.path.join(DATA_PROCESSED_DIR, "test.csv")

train_out.to_csv(train_path, index=False)
val_out.to_csv(val_path, index=False)
test_out.to_csv(test_path, index=False)

print("Deduplicated total rows:", df.shape[0])
print("Saved:")
print(train_path, train_out.shape)
print(val_path, val_out.shape)
print(test_path, test_out.shape)

In [ ]:
import os
import hashlib
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

train_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "train.csv"), usecols=["full_text", "label"])
val_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "val.csv"), usecols=["full_text", "label"])
test_df = pd.read_csv(os.path.join(DATA_PROCESSED_DIR, "test.csv"), usecols=["full_text", "label"])

def md5_hash(s):
    return hashlib.md5(s.encode("utf-8", errors="ignore")).hexdigest()

train_hash = set(train_df["full_text"].astype(str).map(md5_hash))
val_hash = set(val_df["full_text"].astype(str).map(md5_hash))
test_hash = set(test_df["full_text"].astype(str).map(md5_hash))

print("Overlap train vs val:", len(train_hash.intersection(val_hash)))
print("Overlap train vs test:", len(train_hash.intersection(test_hash)))
print("Overlap val vs test:", len(val_hash.intersection(test_hash)))